# Beam Profiles — Side-by-Side Comparison

Compares the four Photonator beam types at the launch plane (z = 0):

| Beam | Class | Key parameter |
|---|---|---|
| Gaussian | `GaussianBeam` | w₀ = 1 mm, divergence = 0.75 mrad |
| Laguerre-Gaussian LG₀¹ | `LaguerreGaussianBeam` | p=0, l=1 (donut mode), w₀ = 1 mm |
| Bessel J₀ | `BesselBeam` | k_r = 3000 m⁻¹, θ_cone = 1 mrad, aperture = 5 mm |
| Airy (2D separable) | `AiryBeam` | x₀ = y₀ = 0.5 mm, extent = ±4 x₀ |

All positions are sampled at z = 0 (the beam waist / aperture plane).
Theoretical intensity profiles are overlaid on the radial plots.

**Reference**: `docs/physics.md §4–7` for beam equations.

## 1. Imports

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import airy, j0

from photonator.beam.airy import AiryBeam
from photonator.beam.bessel import BesselBeam
from photonator.beam.gaussian import GaussianBeam
from photonator.beam.laguerre import LaguerreGaussianBeam

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
RNG = np.random.default_rng(42)
N = 500_000   # photons per beam

## 2. Construct beams and sample photon positions

In [ ]:
# ── Gaussian ─────────────────────────────────────────────────────────────────
W0 = 0.001   # 1 mm beam waist
beam_gauss = GaussianBeam(w0_m=W0, half_angle_divergence_rad=0.00075)
b_gauss = beam_gauss.initialize(N, np.random.default_rng(0))

# ── Laguerre-Gaussian LG_0^1 (donut mode) ────────────────────────────────────
beam_lg = LaguerreGaussianBeam(p=0, l=1, w0_m=W0, half_angle_divergence_rad=0.0)
b_lg = beam_lg.initialize(N, np.random.default_rng(1))

# ── Bessel J0 ────────────────────────────────────────────────────────────────
K_R = 3000.0       # radial wavevector → first ring at r = 2.405/K_R ≈ 0.80 mm
APERTURE = 0.005   # 5 mm aperture radius
beam_bessel = BesselBeam(k_r_per_m=K_R, theta_cone_rad=0.001, aperture_radius_m=APERTURE)
b_bessel = beam_bessel.initialize(N, np.random.default_rng(2))

# ── Airy (2D separable) ───────────────────────────────────────────────────────
X0 = 0.0005   # 0.5 mm transverse scale
beam_airy = AiryBeam(x0_m=X0, y0_m=X0, extent_x0=8.0, extent_y0=8.0)
b_airy = beam_airy.initialize(N, np.random.default_rng(3))

batches = [
    (b_gauss,  "Gaussian",         "viridis"),
    (b_lg,     "LG₀¹ (donut)",    "plasma"),
    (b_bessel, "Bessel J₀",        "inferno"),
    (b_airy,   "Airy",             "cividis"),
]

for b, label, _ in batches:
    r = np.sqrt(b.x_m**2 + b.y_m**2)
    mag = np.sqrt(b.ux**2 + b.uy**2 + b.uz**2)
    print(f"{label:<20}  r_rms={r.std()*1e3:.3f} mm   "
          f"uz_mean={b.uz.mean():.6f}   |u|_max_err={abs(mag-1).max():.2e}")

## 3. Transverse intensity maps (2 × 2)

2D histograms of photon positions at the launch plane, log-normalised to reveal
low-intensity side lobes.

In [ ]:
# Plot ranges chosen to show the interesting structure of each beam
plot_ranges = [
    [[-4e-3, 4e-3], [-4e-3, 4e-3]],   # Gaussian: ±4 mm
    [[-4e-3, 4e-3], [-4e-3, 4e-3]],   # LG:       ±4 mm
    [[-6e-3, 6e-3], [-6e-3, 6e-3]],   # Bessel:   ±6 mm (show multiple rings)
    [[-4e-3, 4e-3], [-4e-3, 4e-3]],   # Airy:     ±4 mm
]

fig, axes = plt.subplots(2, 2, figsize=(11, 10))

for ax, (batch, label, cmap), rng in zip(axes.flat, batches, plot_ranges):
    h = ax.hist2d(
        batch.x_m * 1e3, batch.y_m * 1e3,
        bins=120,
        range=[[r[0] * 1e3, r[1] * 1e3] for r in rng],
        norm=mcolors.LogNorm(),
        cmap=cmap,
    )
    plt.colorbar(h[3], ax=ax, label="Photon count")
    ax.set_xlabel("x  (mm)")
    ax.set_ylabel("y  (mm)")
    ax.set_title(label)
    ax.set_aspect("equal")

fig.suptitle("Transverse intensity maps at z = 0", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Radial intensity profiles with theoretical curves

The radial photon density (photons per unit area) is the area-normalised histogram
$\rho(r) = \frac{dN}{2\pi\,r\,dr}$. This is compared against the analytical intensity
profile for each beam type.

| Beam | Analytical profile $I(r)$ |
|---|---|
| Gaussian | $\exp(-2r^2/w_0^2)$ |
| LG₀¹ | $(r/w_0)^2 \exp(-2r^2/w_0^2)$ (donut) |
| Bessel J₀ | $J_0(k_r\,r)^2$ |
| Airy (1D marginal) | $\mathrm{Ai}(x/x_0)^2$ |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# ── Gaussian ──────────────────────────────────────────────────────────────────
ax = axes[0, 0]
r = np.sqrt(b_gauss.x_m**2 + b_gauss.y_m**2) * 1e3   # mm
r_max = 6.0   # mm
bins = np.linspace(0, r_max, 100)
counts, edges = np.histogram(r, bins=bins)
r_mid = 0.5 * (edges[:-1] + edges[1:])
dr = edges[1] - edges[0]
density = counts / (2 * np.pi * r_mid * dr * N)
ax.plot(r_mid, density / density.max(), color="steelblue", lw=1.5, label="MC")
r_th = np.linspace(0, r_max, 300)
I_gauss = np.exp(-2 * (r_th * 1e-3)**2 / W0**2)
ax.plot(r_th, I_gauss / I_gauss.max(), "k--", lw=1.2, label=r"$\exp(-2r^2/w_0^2)$")
ax.set_xlabel("r  (mm)"); ax.set_ylabel("Normalised intensity")
ax.set_title("Gaussian"); ax.legend(); ax.set_xlim(0, r_max)

# ── LG₀¹ ──────────────────────────────────────────────────────────────────────
ax = axes[0, 1]
r = np.sqrt(b_lg.x_m**2 + b_lg.y_m**2) * 1e3
counts, edges = np.histogram(r, bins=bins)
density = counts / (2 * np.pi * r_mid * dr * N)
ax.plot(r_mid, density / density.max(), color="mediumpurple", lw=1.5, label="MC")
rho = np.sqrt(2) * r_th * 1e-3 / W0
I_lg = rho**2 * np.exp(-rho**2)   # |l|=1, p=0  → (rho^2 * L0(rho^2))^2 * exp(-rho^2) = rho^2 * exp(-rho^2)
ax.plot(r_th, I_lg / I_lg.max(), "k--", lw=1.2, label=r"$(\rho^2)\,e^{-\rho^2}$")
ax.set_xlabel("r  (mm)"); ax.set_ylabel("Normalised intensity")
ax.set_title("LG₀¹ (donut)"); ax.legend(); ax.set_xlim(0, r_max)

# ── Bessel ─────────────────────────────────────────────────────────────────────
ax = axes[1, 0]
r = np.sqrt(b_bessel.x_m**2 + b_bessel.y_m**2) * 1e3
bins_b = np.linspace(0, APERTURE * 1e3, 120)
r_mid_b = 0.5 * (bins_b[:-1] + bins_b[1:])
dr_b = bins_b[1] - bins_b[0]
counts_b, _ = np.histogram(r, bins=bins_b)
density_b = counts_b / (2 * np.pi * r_mid_b * dr_b * N)
ax.plot(r_mid_b, density_b / density_b.max(), color="darkorange", lw=1.5, label="MC")
r_th_b = np.linspace(0, APERTURE * 1e3, 500)
I_bessel = j0(K_R * r_th_b * 1e-3)**2
ax.plot(r_th_b, I_bessel / I_bessel.max(), "k--", lw=1.2, label=r"$J_0(k_r r)^2$")
# Mark first zero of J0 at k_r*r = 2.405
r_first_zero = 2.405 / K_R * 1e3
ax.axvline(r_first_zero, color="gray", lw=0.8, linestyle=":", label=f"1st zero r={r_first_zero:.2f} mm")
ax.set_xlabel("r  (mm)"); ax.set_ylabel("Normalised intensity")
ax.set_title("Bessel J₀"); ax.legend(fontsize=9); ax.set_xlim(0, APERTURE * 1e3)

# ── Airy (1D marginal in x) ───────────────────────────────────────────────────
ax = axes[1, 1]
x_mm = b_airy.x_m * 1e3
x_lim = beam_airy.x_lim_m * 1e3
bins_a = np.linspace(-x_lim, x_lim, 150)
x_mid = 0.5 * (bins_a[:-1] + bins_a[1:])
counts_a, _ = np.histogram(x_mm, bins=bins_a)
ax.plot(x_mid, counts_a / counts_a.max(), color="seagreen", lw=1.5, label="MC (marginal in x)")
x_th = np.linspace(-x_lim, x_lim, 1000)
ai_vals, *_ = airy(x_th * 1e-3 / X0)
I_airy = ai_vals**2
ax.plot(x_th, I_airy / I_airy.max(), "k--", lw=1.2, label=r"$\mathrm{Ai}(x/x_0)^2$")
ax.axvline(-X0 * 1e3, color="gray", lw=0.8, linestyle=":",
           label=f"x₀ = {X0*1e3:.1f} mm")
ax.set_xlabel("x  (mm)"); ax.set_ylabel("Normalised photon count")
ax.set_title("Airy — 1D marginal in x")
ax.legend(fontsize=9); ax.set_xlim(-x_lim, x_lim)

fig.suptitle("Radial / marginal intensity profiles vs theory", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Direction cosine uz — divergence and cone structure

- **Gaussian** (with 0.75 mrad divergence): uz spread reflects the thin-lens divergence.
- **LG₀¹** (collimated): all photons at uz = 1.
- **Bessel** (θ_cone = 1 mrad): all photons at exactly uz = cos(θ_cone).
- **Airy** (collimated): all photons at uz = 1.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
colors = ["steelblue", "mediumpurple", "darkorange", "seagreen"]

for ax, (batch, label, _), color in zip(axes, batches, colors):
    uz_vals = batch.uz
    uz_min, uz_max = uz_vals.min(), uz_vals.max()
    span = max(uz_max - uz_min, 1e-7)
    lo = max(uz_min - 0.05 * span, 1 - 20e-6)
    hi = min(uz_max + 0.05 * span, 1.0 + 1e-9)
    ax.hist(uz_vals, bins=80, range=(lo, hi),
            color=color, edgecolor="none", density=True)
    ax.set_xlabel("uz")
    ax.set_title(label)
    ax.ticklabel_format(axis="x", style="sci", scilimits=(-4, -4), useOffset=True)

axes[0].set_ylabel("Probability density")
fig.suptitle("Direction cosine uz at launch plane", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nuz summary:")
print(f"{'Beam':<22} {'mean uz':>12} {'std uz':>12} {'min uz':>12} {'max uz':>12}")
print("-" * 64)
for batch, label, _ in batches:
    uz = batch.uz
    print(f"{label:<22} {uz.mean():>12.8f} {uz.std():>12.2e} {uz.min():>12.8f} {uz.max():>12.8f}")

## 6. Beam summary statistics

In [ ]:
def beam_stats(batch):
    r = np.sqrt(batch.x_m**2 + batch.y_m**2)
    mag = np.sqrt(batch.ux**2 + batch.uy**2 + batch.uz**2)
    return {
        "r_mean_mm": r.mean() * 1e3,
        "r_rms_mm":  r.std()  * 1e3,
        "uz_mean":   batch.uz.mean(),
        "uz_std":    batch.uz.std(),
        "unit_err":  float(np.abs(mag - 1).max()),
    }

print(f"{'Beam':<22} {'r_mean(mm)':>12} {'r_rms(mm)':>11} {'uz_mean':>10} {'uz_std':>10} {'|u|-1 max':>12}")
print("-" * 82)
for batch, label, _ in batches:
    s = beam_stats(batch)
    print(f"{label:<22} {s['r_mean_mm']:>12.4f} {s['r_rms_mm']:>11.4f} "
          f"{s['uz_mean']:>10.6f} {s['uz_std']:>10.2e} {s['unit_err']:>12.2e}")